### Автоматизация регулярной переоценки и мониторинга конкурентов для гипермаркета 

Цель проекта: Разработать прототип автоматизированного пайплайна для регулярной переоценки ассортимента гипермаркета на основе ценовой политики сети и входящих данных мониторинга конкурентов.   

Задачи:
1. Смоделировать реалистичный синтетический датасет (ассортиментная матрица, правила ценообразования и выгрузка мониторинга с типичными дефектами данных: дубликатами, устаревшими замерами и выбросами).
2. Настроить автоматическую очистку и валидацию входящего мониторинга, разделив регулярные и промо-цены конкурентов.
3. Реализовать алгоритм расчета новых регулярных цен с учетом роли товара (KVI, Обычный, Нишевый), шага ритейл-округления и жесткой защиты минимальной маржи.
4. Оценить экономический эффект переоценки (с учетом базовой эластичности спроса) и сформировать управленческий дашборд в Excel.

### 1 Генерация синтетического датасета

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

In [2]:
# 1. НАШ АССОРТИМЕНТ (500 SKU)

In [3]:
n_sku = 500
categories = ['Молочная продукция', 'Бакалея', 'Напитки', 'Кондитерские изделия', 'Бытовая химия']
roles = ['KVI', 'Обычный', 'Нишевый']

df_our = pd.DataFrame({
    'SKU_ID': [f'SKU_{1000 + i}' for i in range(n_sku)],
    'Название_товара': [f'Товар_{i+1}' for i in range(n_sku)],
    'Категория': np.random.choice(categories, n_sku, p=[0.25, 0.25, 0.2, 0.15, 0.15]),
    'Роль_товара': np.random.choice(roles, n_sku, p=[0.25, 0.55, 0.20]),
    'Закупочная_цена': np.round(np.random.uniform(35, 600, n_sku), 2),
})

df_our['Текущая_цена'] = np.round(df_our['Закупочная_цена'] * np.random.uniform(1.08, 1.48, n_sku), 2)
df_our['Продажи_неделя_шт'] = [
    np.random.randint(250, 1200) if r == 'KVI' else np.random.randint(20, 250)
    for r in df_our['Роль_товара']
]

print(f"Всего товаров в нашем прайсе: {len(df_our)}")
display(df_our.head())

Всего товаров в нашем прайсе: 500


,SKU_ID,Название_товара,Категория,Роль_товара,Закупочная_цена,Текущая_цена,Продажи_неделя_шт
0,SKU_1000,Товар_1,Бакалея,Обычный,139.60,179.75,239
1,SKU_1001,Товар_2,Бытовая химия,Обычный,341.17,433.86,154
2,SKU_1002,Товар_3,Кондитерские изделия,Обычный,528.21,575.88,182
3,SKU_1003,Товар_4,Напитки,Нишевый,448.71,545.86,202
4,SKU_1004,Товар_5,Молочная продукция,Обычный,490.71,604.59,239


In [4]:
# 2. МОНИТОРИНГ КОНКУРЕНТОВ (4 сети)

In [5]:
competitors = ['Ашан', 'ОКЕЙ', 'Перекресток', 'Магнит Семейный']
monitoring_rows = []
base_date = datetime.today()

for _, row in df_our.iterrows():
    if row['Роль_товара'] == 'KVI':
        n_comp = np.random.choice([2, 3, 4], p=[0.2, 0.4, 0.4])
    elif row['Роль_товара'] == 'Обычный':
        n_comp = np.random.choice([0, 1, 2, 3], p=[0.20, 0.35, 0.30, 0.15])
    else:
        n_comp = np.random.choice([0, 1, 2], p=[0.45, 0.35, 0.20])
        
    if n_comp > 0:
        chosen_comps = np.random.choice(competitors, n_comp, replace=False)
        for comp in chosen_comps:
            price_type = np.random.choice(['Регулярная', 'Промо'], p=[0.80, 0.20])
            # Если у конкурента Промо, его цена ниже на 15-30%; если Регулярная — колеблется от -15% до +10%
            if price_type == 'Промо':
                comp_price = np.round(row['Текущая_цена'] * np.random.uniform(0.70, 0.85), 2)
            else:
                comp_price = np.round(row['Текущая_цена'] * np.random.uniform(0.85, 1.10), 2)
                
            days_ago = np.random.randint(0, 11)
            mon_date = (base_date - timedelta(days=days_ago)).strftime('%Y-%m-%d')
            
            monitoring_rows.append({
                'SKU_ID': row['SKU_ID'],
                'Конкурент': comp,
                'Цена_конкурента': comp_price,
                'Тип_цены': price_type,
                'Дата_мониторинга': mon_date
            })

df_comp = pd.DataFrame(monitoring_rows)

# Добавляем реалистичный мусор: 20 выбросов парсера и 15 дублей
anom_idx = np.random.choice(df_comp.index, 20, replace=False)
df_comp.loc[anom_idx[:10], 'Цена_конкурента'] = 1.99
df_comp.loc[anom_idx[10:], 'Цена_конкурента'] *= 10
df_comp = pd.concat([df_comp, df_comp.sample(15)], ignore_index=True)

print(f"Всего строк в сыром мониторинге: {len(df_comp)}")
display(df_comp.head())

Всего строк в сыром мониторинге: 916


,SKU_ID,Конкурент,Цена_конкурента,Тип_цены,Дата_мониторинга
0,SKU_1000,Ашан,1420.20,Промо,2026-11-06
1,SKU_1000,Магнит Семейный,133.08,Промо,2026-11-07
2,SKU_1002,Перекресток,620.13,Регулярная,2026-11-10
3,SKU_1004,ОКЕЙ,5849.00,Регулярная,2026-11-09
4,SKU_1004,Перекресток,591.98,Регулярная,2026-11-07


In [6]:
# 3. ПРАВИЛА ЦЕНОВОЙ ПОЛИТИКИ

In [7]:
df_policy = pd.DataFrame({
    'Роль_товара': ['KVI', 'Обычный', 'Нишевый'],
    'База_сравнения': ['min', 'mean', 'mean'], # KVI сравниваем с МИНИМАЛЬНОЙ ценой рынка, остальные со СРЕДНЕЙ
    'Целевой_индекс_макс': [1.00, 1.02, 1.08],
    'Мин_наценка_проц': [5.0, 15.0, 25.0],
    'Эластичность': [-1.5, -0.8, -0.3] # На сколько % вырастут продажи при снижении цены на 1%
})

display(df_policy)

,Роль_товара,База_сравнения,Целевой_индекс_макс,Мин_наценка_проц,Эластичность
0,KVI,min,1.00,5.0,-1.5
1,Обычный,mean,1.02,15.0,-0.8
2,Нишевый,mean,1.08,25.0,-0.3


In [8]:
df_our.to_csv('our_assortment.csv', index=False, encoding='utf-8-sig')
df_comp.to_csv('competitors_monitoring.csv', index=False, encoding='utf-8-sig')
df_policy.to_csv('pricing_policy.csv', index=False, encoding='utf-8-sig')

In [9]:
# ПЕРВИЧНЫЙ АНАЛИЗ ИСХОДНЫХ ДАННЫХ

In [10]:
print("1. Распределение нашего ассортимента по ролям:")
print(df_our['Роль_товара'].value_counts())
print("\n2. Проверка наценок в нашем прайсе (мин / медиана / макс, %):")
markup = (df_our['Текущая_цена'] - df_our['Закупочная_цена']) / df_our['Закупочная_цена'] * 100
print(f"Мин: {markup.min():.1f}%, Медиана: {markup.median():.1f}%, Макс: {markup.max():.1f}%")
print(f"\n3. Всего строк в мониторинге: {len(df_comp)}")
print("Распределение по типу цены конкурента:")
print(df_comp['Тип_цены'].value_counts())

1. Распределение нашего ассортимента по ролям:
Роль_товара
Обычный    268
KVI        139
Нишевый     93
Name: count, dtype: int64

2. Проверка наценок в нашем прайсе (мин / медиана / макс, %):
Мин: 8.1%, Медиана: 28.4%, Макс: 47.9%

3. Всего строк в мониторинге: 916
Распределение по типу цены конкурента:
Тип_цены
Регулярная    727
Промо         189
Name: count, dtype: int64


### 2. Очистка и валидация данных мониторинга конкурентов

Удаляем технические дубликаты, отсекаем неактуальные замеры (старше 7 дней) и фильтруем аномальные выбросы цен (ошибки сопоставления фасовки и веса). Разделяем регулярные и промо-цены конкурентов без потери данных об акциях.

In [11]:
print(f"Исходных строк в выгрузке: {len(df_comp)}")

# 1. Удаляем полные дубликаты
df_comp_clean = df_comp.drop_duplicates().copy()
print(f"1. После удаления дублей: {len(df_comp_clean)}")

# 2. Оставляем только свежие замеры (не старше 7 полных дней)
df_comp_clean['Дата_мониторинга'] = pd.to_datetime(df_comp_clean['Дата_мониторинга'])
cutoff_date = pd.Timestamp.now().normalize() - pd.Timedelta(days=7)
df_comp_clean = df_comp_clean[df_comp_clean['Дата_мониторинга'] >= cutoff_date]
print(f"2. После отсечения замеров старше 7 дней: {len(df_comp_clean)}")

# 3. Фильтруем ошибки сопоставления (выбросы цены ниже 50% или выше 200% от нашей)
df_comp_clean = df_comp_clean.merge(df_our[['SKU_ID', 'Текущая_цена']], on='SKU_ID', how='left')
df_comp_clean['ratio'] = df_comp_clean['Цена_конкурента'] / df_comp_clean['Текущая_цена']
df_comp_clean = df_comp_clean[(df_comp_clean['ratio'] >= 0.50) & (df_comp_clean['ratio'] <= 2.00)]
print(f"3. После очистки аномальных выбросов: {len(df_comp_clean)}")

Исходных строк в выгрузке: 916
1. После удаления дублей: 901
2. После отсечения замеров старше 7 дней: 658
3. После очистки аномальных выбросов: 642


In [12]:
# Разделяем регулярные и промо-цены конкурентов
reg_comp = df_comp_clean[df_comp_clean['Тип_цены'] == 'Регулярная']
promo_comp = df_comp_clean[df_comp_clean['Тип_цены'] == 'Промо']

# Агрегируем регулярные цены на уровень SKU
reg_agg = reg_comp.groupby('SKU_ID').agg(
    Мин_рег_конкурента=('Цена_конкурента', 'min'),
    Сред_рег_конкурента=('Цена_конкурента', 'mean'),
    Кол_во_рег_конкурентов=('Конкурент', 'count')
).reset_index()
reg_agg['Сред_рег_конкурента'] = reg_agg['Сред_рег_конкурента'].round(2)

# Агрегируем промо-цены на уровень SKU
promo_agg = promo_comp.groupby('SKU_ID').agg(
    Мин_промо_конкурента=('Цена_конкурента', 'min'),
    Кол_во_промо_конкурентов=('Конкурент', 'count')
).reset_index()

print(f"Уникальных товаров с регулярным мониторингом: {len(reg_agg)}")
print(f"Уникальных товаров в промо у конкурентов: {len(promo_agg)}")
display(reg_agg.head())
display(promo_agg.head())

Уникальных товаров с регулярным мониторингом: 317
Уникальных товаров в промо у конкурентов: 114


,SKU_ID,Мин_рег_конкурента,Сред_рег_конкурента,Кол_во_рег_конкурентов
0,SKU_1002,620.13,620.13,1
1,SKU_1004,591.98,591.98,1
2,SKU_1006,523.52,523.52,1
3,SKU_1008,222.58,222.58,1
4,SKU_1011,781.68,795.88,2


,SKU_ID,Мин_промо_конкурента,Кол_во_промо_конкурентов
0,SKU_1000,133.08,1
1,SKU_1004,427.70,1
2,SKU_1009,302.38,2
3,SKU_1014,397.76,1
4,SKU_1025,266.30,1


In [13]:
# Объединяем наш ассортимент с агрегированными данными конкурентов
df_merged = df_our.merge(reg_agg, on='SKU_ID', how='left').merge(promo_agg, on='SKU_ID', how='left')
df_merged['Кол_во_рег_конкурентов'] = df_merged['Кол_во_рег_конкурентов'].fillna(0).astype(int)
df_merged['Кол_во_промо_конкурентов'] = df_merged['Кол_во_промо_конкурентов'].fillna(0).astype(int)

print("Пример объединенной витрины (наш прайс + конкуренты):")
display(df_merged.head())

# Считаем и выводим покрытие ассортимента актуальным мониторингом
coverage = df_merged.groupby('Роль_товара').agg(
    Всего_SKU=('SKU_ID', 'count'),
    Есть_рег_мониторинг=('Кол_во_рег_конкурентов', lambda x: (x > 0).sum()),
    Есть_промо_конкурентов=('Кол_во_промо_конкурентов', lambda x: (x > 0).sum())
)
coverage['Покрытие_рег_%'] = np.round(coverage['Есть_рег_мониторинг'] / coverage['Всего_SKU'] * 100, 1)

print("\nПокрытие ассортимента актуальным мониторингом по ролям:")
display(coverage)

Пример объединенной витрины (наш прайс + конкуренты):


,SKU_ID,Название_товара,Категория,Роль_товара,Закупочная_цена,Текущая_цена,Продажи_неделя_шт,Мин_рег_конкурента,Сред_рег_конкурента,Кол_во_рег_конкурентов,Мин_промо_конкурента,Кол_во_промо_конкурентов
0,SKU_1000,Товар_1,Бакалея,Обычный,139.60,179.75,239,NaN,NaN,0,133.08,1
1,SKU_1001,Товар_2,Бытовая химия,Обычный,341.17,433.86,154,NaN,NaN,0,NaN,0
2,SKU_1002,Товар_3,Кондитерские изделия,Обычный,528.21,575.88,182,620.13,620.13,1,NaN,0
3,SKU_1003,Товар_4,Напитки,Нишевый,448.71,545.86,202,NaN,NaN,0,NaN,0
4,SKU_1004,Товар_5,Молочная продукция,Обычный,490.71,604.59,239,591.98,591.98,1,427.70,1



Покрытие ассортимента актуальным мониторингом по ролям:


,Всего_SKU,Есть_рег_мониторинг,Есть_промо_конкурентов,Покрытие_рег_%
Роль_товара,,,,
KVI,139,132,53,95.0
Нишевый,93,35,11,37.6
Обычный,268,150,50,56.0



### 3. Оценка текущей конкурентоспособности (точка «ДО») и алгоритмическая переоценка

Присоединяем правила ценовой политики к витрине, определяем базу сравнения по каждой роли товара и выявляем позиции, где текущая цена нарушает конкурентоспособность сети.

In [14]:
# 1. Присоединяем правила ценовой политики по ролям товаров
df_calc = df_merged.merge(df_policy, on='Роль_товара', how='left')

pd.set_option('display.max_columns', None)
display(df_calc.head())

,SKU_ID,Название_товара,Категория,Роль_товара,Закупочная_цена,Текущая_цена,Продажи_неделя_шт,Мин_рег_конкурента,Сред_рег_конкурента,Кол_во_рег_конкурентов,Мин_промо_конкурента,Кол_во_промо_конкурентов,База_сравнения,Целевой_индекс_макс,Мин_наценка_проц,Эластичность
0,SKU_1000,Товар_1,Бакалея,Обычный,139.60,179.75,239,NaN,NaN,0,133.08,1,mean,1.02,15.0,-0.8
1,SKU_1001,Товар_2,Бытовая химия,Обычный,341.17,433.86,154,NaN,NaN,0,NaN,0,mean,1.02,15.0,-0.8
2,SKU_1002,Товар_3,Кондитерские изделия,Обычный,528.21,575.88,182,620.13,620.13,1,NaN,0,mean,1.02,15.0,-0.8
3,SKU_1003,Товар_4,Напитки,Нишевый,448.71,545.86,202,NaN,NaN,0,NaN,0,mean,1.08,25.0,-0.3
4,SKU_1004,Товар_5,Молочная продукция,Обычный,490.71,604.59,239,591.98,591.98,1,427.70,1,mean,1.02,15.0,-0.8


In [15]:
# 2. Выбираем базу сравнения с конкурентами: для KVI — минимальную цену рынка, для остальных — среднюю
df_calc['База_цены_конкурента'] = np.where(
    df_calc['База_сравнения'] == 'min',
    df_calc['Мин_рег_конкурента'],
    df_calc['Сред_рег_конкурента']
)

# 3. Считаем текущую наценку (%) и текущий ценовой индекс
df_calc['Текущая_наценка_%'] = np.round(
    (df_calc['Текущая_цена'] - df_calc['Закупочная_цена']) / df_calc['Закупочная_цена'] * 100, 2
)
df_calc['Текущий_индекс_цен'] = np.round(
    df_calc['Текущая_цена'] / df_calc['База_цены_конкурента'], 3
)

# 4. Показываем примеры проблемных товаров-индикаторов (KVI), где мы сильно дороже рынка
print("Примеры товаров-индикаторов (KVI) с критически завышенным ценовым индексом (> 1.15):")
display(
    df_calc[(df_calc['Роль_товара'] == 'KVI') & (df_calc['Текущий_индекс_цен'] > 1.15)]
    [['SKU_ID', 'Категория', 'Закупочная_цена', 'Текущая_цена', 'База_цены_конкурента', 'Текущий_индекс_цен']]
    .head(5)
)

Примеры товаров-индикаторов (KVI) с критически завышенным ценовым индексом (> 1.15):


,SKU_ID,Категория,Закупочная_цена,Текущая_цена,База_цены_конкурента,Текущий_индекс_цен
19,SKU_1019,Бакалея,54.18,78.61,67.70,1.161
39,SKU_1039,Бакалея,213.14,274.96,238.46,1.153
104,SKU_1104,Бытовая химия,597.12,661.47,574.49,1.151
134,SKU_1134,Бытовая химия,421.04,610.87,522.62,1.169
142,SKU_1142,Бакалея,418.08,538.65,466.80,1.154


In [16]:
def smart_round_strict(raw_price, min_allowed_price, direction='down'):
    """Округляет цену по правилам ритейла (.90 до 100 руб., 9.00 от 100 руб.) без пробития мин. маржи."""
    if raw_price < 100:
        rounded = np.floor(raw_price) + 0.90
        if rounded > raw_price and direction == 'down':
            rounded -= 1.00
        while rounded < min_allowed_price:
            rounded += 1.00
    else:
        rounded = np.floor(raw_price / 10) * 10 + 9.00
        if rounded > raw_price and direction == 'down':
            rounded -= 10.00
        while rounded < min_allowed_price:
            rounded += 10.00
    return np.round(rounded, 2)

In [17]:
print(smart_round_strict(64.35, 60.0, 'down'))

63.9


In [18]:
print(smart_round_strict(243.20, 240.0, 'down'))

249.0


In [19]:
def price_engine(row):
    """Определяет новую регулярную цену по 3 приоритетам: защита маржи -> слепая зона -> рынок."""
    # Считаем минимально разрешенную цену ("Пол") с крошечным запасом в 1 копейку на округление
    min_margin_price = row['Закупочная_цена'] * (1 + row['Мин_наценка_проц'] / 100) + 0.01
    curr_p = row['Текущая_цена']
    
    # ШАГ 1: Жесткая защита минимальной наценки
    if curr_p < min_margin_price:
        new_p = smart_round_strict(min_margin_price, min_margin_price, direction='up')
        status = 'Повышение (защита маржи без рынка)' if pd.isna(row['База_цены_конкурента']) else 'Повышение (защита маржи)'
        return pd.Series([new_p, status])
        
    # ШАГ 2: Если конкурентов нет — не трогаем цену
    if pd.isna(row['База_цены_конкурента']):
        return pd.Series([curr_p, 'Нет данных мониторинга'])
    
    # ШАГ 3: Сравниваем нашу цену с допустимым потолком рынка
    max_market_price = row['База_цены_конкурента'] * row['Целевой_индекс_макс']
    
    if curr_p > max_market_price:
        hit_margin_floor = min_margin_price >= max_market_price
        target_p = max(max_market_price, min_margin_price)
        new_p = smart_round_strict(target_p, min_margin_price, direction='down')
        
        if new_p >= curr_p:
            return pd.Series([curr_p, 'Без изменений (уперлись в мин. маржу)'])
            
        if hit_margin_floor or new_p > max_market_price:
            return pd.Series([new_p, 'Снижение (уперлись в мин. маржу)'])
        return pd.Series([new_p, 'Снижение до рынка'])
        
    return pd.Series([curr_p, 'Цена в рынке'])

In [20]:
# Запускаем переоценку на всей витрине
df_calc[['Новая_цена', 'Статус_решения']] = df_calc.apply(price_engine, axis=1)

# Фиксируем промо-активность конкурентов дешевле нашей новой цены
df_calc['Сигнал_промо_конкурентов'] = np.where(
    (df_calc['Мин_промо_конкурента'].notna()) & (df_calc['Мин_промо_конкурента'] < df_calc['Новая_цена']),
    'Конкурент в промо дешевле нас',
    'Ок'
)

# Пересчитываем итоговую наценку и новый ценовой индекс
df_calc['Новая_наценка_%'] = np.round(
    (df_calc['Новая_цена'] - df_calc['Закупочная_цена']) / df_calc['Закупочная_цена'] * 100, 2
)
df_calc['Новый_индекс_цен'] = np.round(
    df_calc['Новая_цена'] / df_calc['База_цены_конкурента'], 3
)

# Показываем пример работы алгоритма на KVI-товарах
display(
    df_calc[df_calc['Роль_товара'] == 'KVI']
    [['SKU_ID', 'Текущая_цена', 'База_цены_конкурента', 'Новая_цена', 'Статус_решения', 'Сигнал_промо_конкурентов']]
    .head(5)
)

,SKU_ID,Текущая_цена,База_цены_конкурента,Новая_цена,Статус_решения,Сигнал_промо_конкурентов
5,SKU_1005,504.75,NaN,504.75,Нет данных мониторинга,Ок
14,SKU_1014,502.30,442.09,459.00,Снижение (уперлись в мин. маржу),Конкурент в промо дешевле нас
15,SKU_1015,604.08,578.40,589.00,Снижение (уперлись в мин. маржу),Ок
19,SKU_1019,78.61,67.70,66.90,Снижение до рынка,Ок
28,SKU_1028,301.67,266.92,259.00,Снижение до рынка,Ок


In [21]:
# КОНТРОЛЬ СОБЛЮДЕНИЯ ЦЕНОВОЙ ПОЛИТИКИ (QA)
violations = df_calc[df_calc['Новая_наценка_%'] < df_calc['Мин_наценка_проц']]
wrong_dir_down = df_calc[(df_calc['Статус_решения'].str.startswith('Снижение')) & (df_calc['Новая_цена'] >= df_calc['Текущая_цена'])]
wrong_dir_up = df_calc[(df_calc['Статус_решения'].str.startswith('Повышение')) & (df_calc['Новая_цена'] <= df_calc['Текущая_цена'])]

print(f"Нарушений минимальной наценки: {len(violations)}")
print(f"Ошибок направления (снижение): {len(wrong_dir_down)}")
print(f"Ошибок направления (повышение): {len(wrong_dir_up)}")

# Сводка принятых решений в виде аккуратной таблицы
decisions_summary = df_calc['Статус_решения'].value_counts().reset_index()
decisions_summary.columns = ['Статус_решения', 'Количество_SKU']
decisions_summary['Доля_%'] = np.round(decisions_summary['Количество_SKU'] / len(df_calc) * 100, 1)

print("\nСтруктура решений по итогам переоценки:")
display(decisions_summary)

Нарушений минимальной наценки: 0
Ошибок направления (снижение): 0
Ошибок направления (повышение): 0

Структура решений по итогам переоценки:


,Статус_решения,Количество_SKU,Доля_%
0,Снижение до рынка,143,28.6
1,Нет данных мониторинга,137,27.4
2,Цена в рынке,93,18.6
3,Повышение (защита маржи без рынка),46,9.2
4,Повышение (защита маржи),40,8.0
5,Снижение (уперлись в мин. маржу),37,7.4
6,Без изменений (уперлись в мин. маржу),4,0.8



### 4. Оценка экономического эффекта (с учетом эластичности спроса) и выгрузка витрины

Рассчитываем прогнозное изменение объема продаж при изменении цены, оцениваем влияние переоценки на выручку и валовую маржу сети, формируем списки сигналов для смежных отделов и экспортируем итоговую таблицу в Excel.


In [22]:
# 1. Считаем процентное изменение цены
df_calc['Изменение_цены_%'] = (df_calc['Новая_цена'] - df_calc['Текущая_цена']) / df_calc['Текущая_цена']

# 2. Прогнозируем изменение спроса (%) с учетом коэффициента эластичности
# Ограничиваем изменение спроса разумным коридором от -40% до +60%
raw_demand_change = df_calc['Изменение_цены_%'] * df_calc['Эластичность']
df_calc['Прогноз_изменения_спроса_%'] = np.clip(raw_demand_change, -0.40, 0.60)

df_calc['Прогноз_продаж_шт'] = np.round(
    df_calc['Продажи_неделя_шт'] * (1 + df_calc['Прогноз_изменения_спроса_%'])
).astype(int)

# 3. Считаем выручку и валовую маржу (ДО и ПОСЛЕ) в рублях
df_calc['Текущая_выручка_руб'] = np.round(df_calc['Текущая_цена'] * df_calc['Продажи_неделя_шт'], 2)
df_calc['Прогноз_выручка_руб'] = np.round(df_calc['Новая_цена'] * df_calc['Прогноз_продаж_шт'], 2)

df_calc['Текущая_маржа_руб'] = np.round((df_calc['Текущая_цена'] - df_calc['Закупочная_цена']) * df_calc['Продажи_неделя_шт'], 2)
df_calc['Прогноз_маржа_руб'] = np.round((df_calc['Новая_цена'] - df_calc['Закупочная_цена']) * df_calc['Прогноз_продаж_шт'], 2)
df_calc['Прирост_маржи_руб'] = np.round(df_calc['Прогноз_маржа_руб'] - df_calc['Текущая_маржа_руб'], 2)

# 4. Формируем сигнал для отдела закупок (товары, где цена уперлась в мин. маржу и осталась выше рынка)
df_calc['Сигнал_закупкам'] = np.where(
    (df_calc['Новый_индекс_цен'].notna()) & (df_calc['Новый_индекс_цен'] > df_calc['Целевой_индекс_макс']),
    'Требует снижения закупки',
    'В норме'
)

In [23]:
# Сводная экономика по ролям товаров
summary_econ = df_calc.groupby('Роль_товара').agg(
    SKU=('SKU_ID', 'count'),
    Ср_индекс_ДО=('Текущий_индекс_цен', 'mean'),
    Ср_индекс_ПОСЛЕ=('Новый_индекс_цен', 'mean'),
    Выручка_ДО_руб=('Текущая_выручка_руб', 'sum'),
    Выручка_ПОСЛЕ_руб=('Прогноз_выручка_руб', 'sum'),
    Маржа_ДО_руб=('Текущая_маржа_руб', 'sum'),
    Маржа_ПОСЛЕ_руб=('Прогноз_маржа_руб', 'sum'),
    Дельта_маржи_руб=('Прирост_маржи_руб', 'sum')
).round(2)

print("Итоговый экономический эффект переоценки по ролям товаров (за неделю):")
display(summary_econ)

print("\nСВОДНЫЕ ПОКАЗАТЕЛИ ПО СЕТИ (ЗА НЕДЕЛЮ)")
rev_before = df_calc['Текущая_выручка_руб'].sum()
rev_after = df_calc['Прогноз_выручка_руб'].sum()
margin_before = df_calc['Текущая_маржа_руб'].sum()
margin_after = df_calc['Прогноз_маржа_руб'].sum()

print(f"Выручка ДО:        {rev_before:,.2f} руб.")
print(f"Выручка ПОСЛЕ:     {rev_after:,.2f} руб.")
print(f"Прирост выручки:   {rev_after - rev_before:+,.2f} руб. ({(rev_after - rev_before)/rev_before*100:+.2f}%)")
print("-" * 45)
print(f"Валовая маржа ДО:    {margin_before:,.2f} руб. (рентабельность {margin_before/rev_before*100:.1f}%)")
print(f"Валовая маржа ПОСЛЕ: {margin_after:,.2f} руб. (рентабельность {margin_after/rev_after*100:.1f}%)")
print(f"Изменение маржи:     {margin_after - margin_before:+,.2f} руб.")
print("-" * 45)
print(f"Товаров в промо у конкурентов дешевле нас: {(df_calc['Сигнал_промо_конкурентов'] != 'Ок').sum()} шт.")
print(f"Товаров для переговоров с поставщиками:    {(df_calc['Сигнал_закупкам'] != 'В норме').sum()} шт.")


Итоговый экономический эффект переоценки по ролям товаров (за неделю):


,SKU,Ср_индекс_ДО,Ср_индекс_ПОСЛЕ,Выручка_ДО_руб,Выручка_ПОСЛЕ_руб,Маржа_ДО_руб,Маржа_ПОСЛЕ_руб,Дельта_маржи_руб
Роль_товара,,,,,,,,
KVI,139,1.07,0.99,45759541.12,46763596.89,10025178.69,7617612.17,-2407566.52
Нишевый,93,1.03,1.07,4993063.00,5119033.76,1052543.51,1229558.62,177015.11
Обычный,268,1.04,1.01,15486326.00,15444158.61,3449797.83,3352066.13,-97731.70



СВОДНЫЕ ПОКАЗАТЕЛИ ПО СЕТИ (ЗА НЕДЕЛЮ)
Выручка ДО:        66,238,930.12 руб.
Выручка ПОСЛЕ:     67,326,789.26 руб.
Прирост выручки:   +1,087,859.14 руб. (+1.64%)
---------------------------------------------
Валовая маржа ДО:    14,527,520.03 руб. (рентабельность 21.9%)
Валовая маржа ПОСЛЕ: 12,199,236.92 руб. (рентабельность 18.1%)
Изменение маржи:     -2,328,283.11 руб.
---------------------------------------------
Товаров в промо у конкурентов дешевле нас: 114 шт.
Товаров для переговоров с поставщиками:    69 шт.


In [24]:
# Сохраняем итоговую витрину в Excel
df_calc.to_excel('lenta_pricing_final_data.xlsx', index=False, sheet_name='Витрина_данных')
print("\nИтоговая витрина успешно сохранена в файл 'lenta_pricing_final_data.xlsx'.")


Итоговая витрина успешно сохранена в файл 'lenta_pricing_final_data.xlsx'.


### 5. Итоговые выводы и рекомендации для бизнеса

По итогам автоматической переоценки 500 SKU (с учетом актуального мониторинга 4 федеральных сетей и ценовой политики):

1. **Восстановление ценового имиджа (KVI):** Средний ценовой индекс по товарам-индикаторам снижен с **1.070** до целевого уровня **0.990** (на 1% дешевле минимального конкурента). За счет высокой эластичности спроса на KVI прогнозируется рост недельной выручки в сегменте на **+1.00 млн руб.** (с 45.76 до 46.76 млн руб.) при плановой инвестиции **2.41 млн руб.** валовой маржи в трафик.
2. **Компенсация маржи за счет нишевого ассортимента:** Корректировка заниженных наценок на неэластичных нишевых товарах (индекс скорректирован со **1.030** до **1.070**) приносит **+177.0 тыс. руб.** дополнительной недельной маржи без потери оборота.
3. **Задачи для отдела закупок (69 SKU):** Выявлено 69 позиций (13.8% ассортимента), где даже при снижении цены до минимально допустимой наценки мы остаемся дороже рынка. Рекомендуется передать данный список категорийным менеджерам для пересмотра закупочных цен с поставщиками.
4. **Задачи для промо-отдела (114 SKU):** По 114 позициям конкуренты проводят краткосрочные промо-акции ниже нашей новой регулярной цены. Регулярные цены по ним сохранены во избежание разрушения базовой маржи; Рекомендуется передать данный список в промо-отдел и категорийным менеджерам для оценки необходимости ответной акции и согласования временных промо-условий с поставщиками.
5. **Работа со «слепой зоной» мониторинга (137 SKU):** 27.4% ассортимента не имеют актуальных замеров. Часть из них (нишевые товары и собственные торговые марки) объективно отсутствует у конкурентов и не требует прямого парсинга. Однако брендовые позиции из сегментов KVI (7 SKU) и Обычный (80+ SKU), попавшие в этот список, рекомендовано передать в отдел мониторинга для проверки сопоставления (матчинга) и постановки на регулярный контроль.